# Explainable Commodity Prices — Results Orchestrator

**Audience:** senior economics staff reviewing the full analysis.

This notebook surfaces the headline results from the full pipeline (vanilla AE baseline + β-VAE robustness). Implementation detail lives in `reports/` and the `src/eqcp` pipelines.

### Pipeline map
| Stage | Command | Output |
|-------|---------|--------|
| Overall macro spanning | `make mapping` / `make mapping-beta` | `results/macro_mapping[_beta]/` |
| Sector decomposition | `make sectors` / `make sectors-beta` | `results/sector_analysis[_beta]/` |
| Forecast attribution | `make forecast` / `make forecast-beta` | `results/forecast_pbsv[_beta]/` |

**Panel:** 21 daily commodity futures. **Macro:** 37 stationary variables. **Factors:** K=5. **Seed:** 0.



## 0. Setup & run pipelines


In [ ]:
from __future__ import annotations

import subprocess
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})
SPANNED_C, WEAK_C, NEUTRAL_C = "#2A6F97", "#C44E52", "#8C8C8C"
SECTOR_C = {"agriculture": "#55A868", "energy": "#DD8452", "metals": "#8172B3"}

# Resolve project root and make `eqcp` importable without a separate pip install.
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path(__file__).resolve().parent if "__file__" in dir() else ROOT
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Open/run this notebook with the working directory set to the project root "
        "(explainable_commodity_prices/)."
    )
_SRC = ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from eqcp.config import load_factor_model_config

factor_cfg = load_factor_model_config()
print(f"Project root: {ROOT}")
print(f"Factor model: K={factor_cfg.n_factors}, vanilla activation={factor_cfg.activation}, beta={factor_cfg.beta}")

def run(cmd: str) -> None:
    print(f"\n>>> {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=ROOT)

# Uncomment to regenerate all artifacts (slow: ~15-30 min total)
# run("make mapping && make mapping-beta && make sectors && make sectors-beta && make forecast && make forecast-beta")




## 1. Dimension analysis — overall macro spanning

**Question:** Of the K=5 latent commodity factors, how many directions are *genuinely* spanned by the macro panel out-of-sample?

**Verdict rule:** OOS canonical correlation ρ > 0.3 **and** circular-shift permutation p < 0.05.


In [ ]:
cc = pd.read_csv(ROOT / "results/macro_mapping/canonical_correlations.csv", index_col=0)
cc["spanned"] = (cc["rho_oos"] > 0.3) & (cc["perm_p"] < 0.05)
n_spanned = int(cc["spanned"].sum())
print(f"{n_spanned} of {len(cc)} factor directions are macro-spanned out-of-sample.\n")
display(cc[["rho_insample", "rho_oos", "perm_p", "spanned"]].round(3))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
colors = [SPANNED_C if s else WEAK_C for s in cc["spanned"]]
ax.bar(cc.index, cc["rho_oos"], color=colors)
ax.plot(cc.index, cc["null_p95"], "k--o", lw=1, label="permutation null (p95)")
ax.axhline(0.3, color=NEUTRAL_C, ls=":", label="spanning threshold (ρ=0.3)")
ax.set_ylim(-0.05, 1.0)
ax.set_ylabel("OOS canonical correlation")
ax.set_title("Overall macro spanning: which latent directions are macro-recoverable?")
ax.legend()
plt.tight_layout(); plt.show()

loadings = pd.read_csv(ROOT / "results/macro_mapping/canonical_loadings_macro.csv", index_col=0)
for dim in cc.index[cc["spanned"]]:
    top = loadings[dim].abs().sort_values(ascending=False).head(5)
    drivers = ", ".join(f"{v}({loadings.loc[v, dim]:+.2f})" for v in top.index)
    print(f"{dim}: top macro drivers → {drivers}")



## 2. Sector-wise mapping (vanilla vs β-VAE)

**Question:** Is the macro-spanning story uniform across sectors — and is it stable under β-VAE?

We expect **energy** to be the most macro-linked and **agriculture** the most idiosyncratic; the β-VAE run checks that ranking is not an AE artefact.



In [ ]:
def _sector_spanned_counts(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    return df.groupby("block")["spanned"].sum().astype(int)

vanilla_path = ROOT / "results/sector_analysis/sector_cca_summary.csv"
beta_path = ROOT / "results/sector_analysis_beta/sector_cca_summary.csv"

sector_v = pd.read_csv(vanilla_path)
print("Vanilla AE — sector spanning summary")
display(sector_v.round(3))

if beta_path.exists():
    sector_b = pd.read_csv(beta_path)
    print("\nβ-VAE — sector spanning summary")
    display(sector_b.round(3))

    cmp = pd.DataFrame({
        "block": _sector_spanned_counts(vanilla_path).index,
        "n_spanned_vanilla": _sector_spanned_counts(vanilla_path).to_numpy(),
        "n_spanned_beta": _sector_spanned_counts(beta_path).reindex(
            _sector_spanned_counts(vanilla_path).index
        ).fillna(0).astype(int).to_numpy(),
    })
    cmp["match"] = cmp["n_spanned_vanilla"] == cmp["n_spanned_beta"]
    print("\nSpanned-direction counts by block (vanilla vs β-VAE):")
    display(cmp)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, df, title in zip(axes, [sector_v, sector_b], ["Vanilla AE", "β-VAE"]):
        counts = df.groupby("block")["spanned"].sum().reindex(
            ["overall", "energy", "agriculture", "metals"]
        ).fillna(0)
        ax.bar(counts.index, counts.to_numpy(), color=SPANNED_C)
        for i, v in enumerate(counts.to_numpy()):
            ax.text(i, v + 0.05, str(int(v)), ha="center", fontsize=10)
        ax.set_ylabel("# macro-spanned directions")
        ax.set_title(title)
    fig.suptitle("Sector-wise macro spanning — factor model comparison", y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("Run `make sectors-beta` for β-VAE sector results.")

for label, subdir in [("Vanilla", "sector_analysis"), ("β-VAE", "sector_analysis_beta")]:
    fig = ROOT / "figures" / subdir / "sector_spanning_bars.png"
    if fig.exists():
        print(f"\n{label} sector spanning bars:")
        display(Image(filename=str(fig)))



## 3. Factor model comparison — Vanilla AE vs β-VAE

**Question:** Does the β-VAE (KL-regularised, disentanglement-prior latents) change how many directions are macro-spanned?

β=4.0 penalises redundant latents; we compare OOS spanning counts side-by-side.


In [ ]:
cmp_path = ROOT / "results/macro_mapping/factor_model_comparison.csv"
if cmp_path.exists():
    cmp_df = pd.read_csv(cmp_path)
    display(cmp_df.round(4))
else:
    print("Run `make mapping` first to generate factor_model_comparison.csv")

vanilla_cc = pd.read_csv(ROOT / "results/macro_mapping/canonical_correlations.csv", index_col=0)
beta_cc_path = ROOT / "results/macro_mapping_beta/canonical_correlations.csv"
if beta_cc_path.exists():
    beta_cc = pd.read_csv(beta_cc_path, index_col=0)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, df, title in zip(
        axes,
        [vanilla_cc, beta_cc],
        ["Vanilla AE", "β-VAE"],
    ):
        sp = (df["rho_oos"] > 0.3) & (df["perm_p"] < 0.05)
        ax.bar(df.index, df["rho_oos"], color=[SPANNED_C if s else WEAK_C for s in sp])
        ax.axhline(0.3, color=NEUTRAL_C, ls=":")
        ax.set_title(title)
        ax.set_ylabel("OOS ρ")
    fig.suptitle("Macro spanning by factor model", y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("Run `make mapping-beta` to generate beta-VAE spanning results.")



## 4. Forecast attribution — PBSV (Shapley values)

**Question:** Does macro information add *out-of-sample forecast value* for commodity returns, and which canonical directions contribute?

PBSV attributes the AR(1)-vs-full MSE gain to each macro-linked canonical variate. Negative φ means the direction hurts the forecast.


In [ ]:
shapley = pd.read_csv(ROOT / "results/forecast_pbsv/pbsv_shapley.csv")
h1 = shapley[shapley["horizon"] == 1].copy()
display(h1.round(6))

fig_path = ROOT / "figures/forecast_pbsv/pbsv_per_commodity.png"
if fig_path.exists():
    display(Image(filename=str(fig_path)))

pooled = pd.read_csv(ROOT / "results/forecast_pbsv/forecast_accuracy_pooled.csv")
headline = pooled[pooled["horizon"] == 1].iloc[0]
print(
    f"h=1 pooled R²_OOS(full vs AR1) = {headline['r2_pool_std_vs_ar1']:+.5f}  |  "
    f"Clark-West p = {headline['cw_pool_p']:.4f}  |  "
    f"gate_passed = {bool(headline['gate_passed'])}"
)



In [ ]:
# β-VAE forecast comparison (optional — requires `make forecast-beta`)
beta_pooled_path = ROOT / "results/forecast_pbsv_beta/forecast_accuracy_pooled.csv"
if beta_pooled_path.exists():
    beta_pooled = pd.read_csv(beta_pooled_path)
    beta_h1 = beta_pooled[beta_pooled["horizon"] == 1].iloc[0]
    compare = pd.DataFrame(
        {
            "model": ["vanilla AE", "β-VAE"],
            "r2_oos_vs_ar1": [headline["r2_pool_std_vs_ar1"], beta_h1["r2_pool_std_vs_ar1"]],
            "cw_p": [headline["cw_pool_p"], beta_h1["cw_pool_p"]],
            "gate_passed": [bool(headline["gate_passed"]), bool(beta_h1["gate_passed"])],
        }
    )
    print("Forecast null is stable across factor models:")
    display(compare.round(5))
else:
    print("Run `make forecast-beta` for the β-VAE forecast comparison.")


## 5. Forecast horizon analysis

**Question:** Does macro transmission strengthen at longer horizons (weekly / monthly / quarterly)?

We report pooled OOS accuracy and grouped PBSV (spanned vs weak blocks) at h ∈ {1, 5, 21, 63} trading days.


In [ ]:
trans = pd.read_csv(ROOT / "results/forecast_pbsv/macro_transmission_by_horizon.csv")
cols = [
    "horizon", "r2_oos_vs_ar1", "cw_p_overlap", "phi_spanned", "phi_weak",
    "retained_share_spanned", "gate_passed",
]
display(trans[cols].round(5))

fig_path = ROOT / "figures/forecast_pbsv/transmission_by_horizon.png"
if fig_path.exists():
    display(Image(filename=str(fig_path)))

grouped = pd.read_csv(ROOT / "results/forecast_pbsv/pbsv_grouped.csv")
print("\nGrouped PBSV by horizon (spanned vs weak blocks):")
display(grouped.round(6))



## 6. Takeaways

1. **Compression works:** ~5 latent factors summarise 21 commodity return series; **~2 directions** are macro-spanned OOS (dollar/inflation/energy-equity and dollar/commodity-FX).
2. **Sector heterogeneity:** Energy is the most macro-linked sector; agriculture the most idiosyncratic — **stable under β-VAE** (see §2 comparison table).
3. **β-VAE robustness:** Overall spanning count matches vanilla (2/5); sector spanned counts should agree block-by-block if the macro story is real.
4. **Forecast null:** Macro-linked factors explain contemporaneous co-movement but do **not** pass the share-of-gain gate at any horizon — consistent with near-efficient daily futures.
5. **Horizon ladder:** Longer horizons do not rescue forecast value (§5); gate fails at h ∈ {1, 5, 21, 63} for both factor models.

**Full reports:** `reports/macro_mapping_report.md`, `reports/sector_analysis_report.md`, `reports/sector_analysis_beta_report.md`, `reports/forecast_pbsv_report.md`.

